In [1]:
import pandas as pd
import nltk
import nltk.corpus
import nltk.tokenize
import re
import pymorphy2
import locale
import numpy as np
import gensim
from string import punctuation
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec, FastText
from gensim.models.fasttext import load_facebook_model
import torch
from torch import nn
from torch.utils.data import DataLoader
import torch.optim as optim

#from rus_preprocessing_udpipe import num_replace, clean_token, clean_lemma, list_replace, unify_sym, process
import sys
import os
import wget
import re
import random
#from ufal.udpipe import Model, Pipeline

In [2]:
tqdm.pandas()

In [3]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\den26\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\den26\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
data_directory = "../Data"

In [5]:
negative_file = os.path.join(data_directory, "negative.csv")
positive_file = os.path.join(data_directory, "positive.csv")
header_names = ["id", "tdate", "tmane",'ttext', 'ttype', 'trep', 'trtv','tfav', 'tstcount', 'tfol', 'tfrien', 'listcount']

negative_df = pd.read_csv(negative_file, delimiter=";",header=None, names = header_names)
positive_df = pd.read_csv(positive_file, delimiter=";",header=None, names = header_names)
negative_df = negative_df.sample(5000, random_state = 42)
positive_df = positive_df.sample(5000, random_state = 42)
df = pd.concat([negative_df, positive_df])

In [6]:
dictionary_file = os.path.join(data_directory, "words_all_full_rating.csv")

dictionary = pd.read_csv(dictionary_file, delimiter = ";", encoding = "windows-1251")
dictionary["mean"] = dictionary["mean"].apply(lambda x: x.replace(',', '.'))
dictionary["dispersion"] = dictionary["dispersion"].apply(lambda x: x.replace(',', '.'))

In [7]:
# punctuations = set(punctuation)
# punctuations.update(['``','...',"''",'«','»','…','”','”','“','-','–','..'])
stopwords = nltk.corpus.stopwords.words("russian")
stopwords.remove("не")
stopwords.remove("нет")
stopwords.remove("ни")
stopwords.remove("хорошо")
stopwords.remove("больше")
stopwords.remove("лучше")
stopwords.remove("нельзя")
stopwords.remove("никогда")
stopwords.remove("можно")

morph = pymorphy2.MorphAnalyzer()

def preprocess(text):
    text = re.sub("[^а-яА-ЯёЁ ]", ' ', text)
    text_tokenized = nltk.word_tokenize(text)
    text_tokenized = [x.lower() for x in text_tokenized if x not in stopwords]
    text_tokenized = [morph.parse(word)[0].normal_form for word in text_tokenized]
    return text_tokenized

In [8]:
df["ttext_preprocessed"] = df["ttext"].progress_apply(preprocess)

100%|██████████████████████████████████| 10000/10000 [00:09<00:00, 1031.93it/s]


In [9]:
words_set = set()
for row in df["ttext_preprocessed"]:
    words_set = set.union(words_set, set(row))
    
num_embeddings = len(words_set)

In [10]:
ttext_preprocessed = df["ttext_preprocessed"]
word2vec = Word2Vec(sentences=ttext_preprocessed, vector_size=300, window=5, min_count=1, workers=4)
word2vec.build_vocab(ttext_preprocessed)
word2vec.train(ttext_preprocessed, total_examples=word2vec.corpus_count, epochs=300, report_delay=1)
word2vec.init_sims(replace=True)

C:\Users\den26\AppData\Local\Temp/ipykernel_15112/71021739.py:5: DeprecationWarning: Call to deprecated `init_sims` (Gensim 4.0.0 implemented internal optimizations that make calls to init_sims() unnecessary. init_sims() is now obsoleted and will be completely removed in future versions. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  word2vec.init_sims(replace=True)


In [11]:
word2vec_df = df[["ttext_preprocessed"]]
word2vec_df["ttext_vectors"] = ttext_preprocessed.progress_apply(lambda x: pd.Series(x).apply(lambda x: word2vec.wv[x]).array)
word2vec_df["ttext_vectors"] = word2vec_df["ttext_vectors"].progress_apply(lambda x: np.mean(x))
word2vec_df["ttype"] = df["ttype"].apply(lambda x: 1 if x == 1 else 0)

100%|██████████████████████████████████| 10000/10000 [00:01<00:00, 8098.51it/s]
C:\Users\den26\AppData\Local\Temp/ipykernel_15112/104287943.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  word2vec_df["ttext_vectors"] = ttext_preprocessed.progress_apply(lambda x: pd.Series(x).apply(lambda x: word2vec.wv[x]).array)
100%|█████████████████████████████████| 10000/10000 [00:00<00:00, 29454.46it/s]
C:\Users\den26\AppData\Local\Temp/ipykernel_15112/104287943.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  w

In [12]:
fasttext = FastText(vector_size=300, window=5, min_count=1, workers=4)
fasttext.build_vocab(corpus_iterable = ttext_preprocessed)
fasttext.train(ttext_preprocessed, total_examples=fasttext.corpus_count, epochs=10, report_delay=1)
fasttext.init_sims(replace=True)

C:\Users\den26\AppData\Local\Temp/ipykernel_15112/3947742650.py:4: DeprecationWarning: Call to deprecated `init_sims` (Gensim 4.0.0 implemented internal optimizations that make calls to init_sims() unnecessary. init_sims() is now obsoleted and will be completely removed in future versions. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  fasttext.init_sims(replace=True)


In [141]:
fasttext_df = df[["ttext_preprocessed"]]
fasttext_df["ttext_vectors"] = ttext_preprocessed.progress_apply(lambda x: pd.Series(x).apply(lambda x: fasttext.wv[x]).array)
fasttext_df["ttext_vectors"] = fasttext_df["ttext_vectors"].progress_apply(lambda x: np.mean(x))
fasttext_df["ttype"] = df["ttype"].apply(lambda x: 1 if x == 1 else 0)

100%|██████████████████████████████████| 10000/10000 [00:02<00:00, 4489.04it/s]
C:\Users\den26\AppData\Local\Temp/ipykernel_15112/3504630237.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasttext_df["ttext_vectors"] = ttext_preprocessed.progress_apply(lambda x: pd.Series(x).apply(lambda x: fasttext.wv[x]).array)
100%|██████████████████████████████████| 10000/10000 [00:02<00:00, 4383.40it/s]
C:\Users\den26\AppData\Local\Temp/ipykernel_15112/3504630237.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
 

In [14]:
words = list(words_set)
word_to_idx = {words[i] : i for i in range(len(words))}    

In [142]:
embeddings_df = df[["ttext_preprocessed"]]
embeddings_df["ttext_vectors"] = ttext_preprocessed.apply(lambda x: np.array([word_to_idx[word] for word in x]))
embeddings_df["ttype"] = df["ttype"].apply(lambda x: 1 if x == 1 else 0)
max_embeddings_len = max(embeddings_df["ttext_vectors"].apply(len))
embeddings_df["ttext_vectors"] = embeddings_df["ttext_vectors"].apply(lambda x: x.tolist()).apply(lambda x: np.array(x + [0] * (max_embeddings_len - len(x))))

C:\Users\den26\AppData\Local\Temp/ipykernel_15112/3156013112.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  embeddings_df["ttext_vectors"] = ttext_preprocessed.apply(lambda x: np.array([word_to_idx[word] for word in x]))
C:\Users\den26\AppData\Local\Temp/ipykernel_15112/3156013112.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  embeddings_df["ttype"] = df["ttype"].apply(lambda x: 1 if x == 1 else 0)
C:\Users\den26\AppData\Local\Temp/ipykernel_15112/3156013112.py:5: SettingWithCopyWarning: 
A value 

In [101]:
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, x, y = None):
        self.x = x
        self.y = y
    
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return torch.FloatTensor(self.x.iloc[idx]), torch.LongTensor([self.y.iloc[idx]])

In [102]:
class EmbeddingsDataset(torch.utils.data.Dataset):
    def __init__(self, x, y = None):
        self.x = x
        self.y = y
    
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return torch.LongTensor(self.x.iloc[idx]), torch.LongTensor([self.y.iloc[idx]])

In [17]:
def train_val_test_split(X, y):
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, stratify = y)
    X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size = 0.25, stratify = y_train)
    return X_train, X_val, X_test, y_train, y_val, y_test

In [152]:
def get_data_loaders(dataset_type, X, y, batch_size = 64):
    X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y)
    
    train_ds = dataset_type(X_train, y_train)
    val_ds = dataset_type(X_val, y_val)
    test_ds = dataset_type(X_test, y_test)
    
    train_dl = DataLoader(train_ds, batch_size = batch_size, shuffle = True)
    val_dl = DataLoader(val_ds, batch_size = batch_size, shuffle = True)
    test_dl = DataLoader(test_ds, batch_size = batch_size, shuffle = False)
    
    return train_dl, val_dl, test_dl

In [50]:
class SimpleNetwork(nn.Module):
    def __init__(self, layer_sizes):
        super().__init__()
        self.fcs = [nn.Linear(300, layer_sizes[0])]
        self.fcs.extend([nn.Linear(layer_sizes[i], layer_sizes[i + 1]) for i in range(len(layer_sizes) - 1)])
        self.fc_final = nn.Linear(layer_sizes[len(layer_sizes) - 1], 2)
    
    def forward(self, x):
        for fc in self.fcs:
            x = fc(x)
            x = nn.ReLU()(x)
            
        x = self.fc_final(x)
        x = nn.Softmax(dim = 1)(x)
        return x

In [114]:
class EmbeddingsNetwork(nn.Module):
    def __init__(self, layer_sizes):
        super().__init__()
        self.emb = nn.Embedding(num_embeddings, 300)
        self.fcs = [nn.Linear(300, layer_sizes[0])]
        self.fcs.extend([nn.Linear(layer_sizes[i], layer_sizes[i + 1]) for i in range(len(layer_sizes) - 1)])
        self.fc_final = nn.Linear(layer_sizes[len(layer_sizes) - 1], 2)

        
    def forward(self, x):
        #print(x.shape)
        x = self.emb(x).mean(axis = 1)
        #print(x.shape)
        for fc in self.fcs:
            x = fc(x)
            x = nn.ReLU()(x)
            
        x = self.fc_final(x)
        x = nn.Softmax(dim = 1)(x)
        return x

In [21]:
def set_random_state(random_state):
    torch.manual_seed(random_state)
    random.seed(random_state)
    np.random.seed(random_state)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)
        torch.cuda.manual_seed(random_state)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [27]:
def train_loop(net, train_loader, val_loader, test_loader, num_epochs, learning_rate, metrics, is_logs_on = False):
    set_random_state(42)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):
        if is_logs_on:
            print(f'Epoch [{epoch+1}/{num_epochs}]')
        net.train()
        total_loss = 0
        preds = []
        targets = []

        processed = 0
        for inputs, labels in train_loader:
            processed += 64
            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, labels.reshape(len(labels)))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds.extend(torch.max(outputs, 1).indices.tolist())
            targets.extend(labels.reshape(len(labels)).tolist())

        avg_loss = total_loss / len(train_loader)
        if is_logs_on:
            print('Train:')
            print(f'\t Loss: {avg_loss:.4f}')
            for metric_name, metric, kwargs in metrics:
                print(f'\t {metric_name}: {metric(targets, preds, **kwargs):.4f}')
    
        with torch.no_grad():
            total_loss = 0
            preds = []
            targets = []

            for inputs, labels in val_loader:
                outputs = net(inputs)
                loss = criterion(outputs, labels.reshape(len(labels)))

                total_loss += loss.item()
                preds.extend(torch.max(outputs, 1).indices.tolist())
                targets.extend(labels.tolist())

            avg_loss = total_loss / len(train_loader)
            if is_logs_on:
                print('Validation:')
                print(f'\t Loss: {avg_loss:.4f}')
                for metric_name, metric, kwargs in metrics:
                    print(f'\t {metric_name}: {metric(targets, preds, **kwargs):.4f}')
    
    with torch.no_grad():
        total_loss = 0
        preds = []
        targets = []
    
        for inputs, labels in test_loader:
            outputs = net(inputs)
            loss = criterion(outputs, labels.reshape(len(labels)))

            total_loss += loss.item()
            preds.extend(torch.max(outputs, 1).indices.tolist())
            targets.extend(labels.tolist())

        avg_loss = total_loss / len(train_loader)
        if is_logs_on:
            print('Test:')
            print(f'\t Loss: {avg_loss:.4f}')
            for metric_name, metric, kwargs in metrics:
                print(f'\t {metric_name}: {metric(targets, preds, **kwargs):.4f}')
        
        return {metric_name: metric(targets, preds, **kwargs) for metric_name, metric, kwargs in metrics}

## Results

In [150]:
def get_results(method_name, net_type, train_dl, val_dl, test_dl, layer_sizes_list, learning_rates_list, epochs_num, metrics):
    results = []
    i_progress = 1
    for layer_sizes in layer_sizes_list:
        for learning_rate in learning_rates_list:
            net = net_type(layer_sizes)
            result = train_loop(net, train_dl, val_dl, test_dl, epochs_num, learning_rate, metrics)
            result["method"] = method_name
            result["layer_sizes"] = layer_sizes
            result["learning_rate"] = learning_rate
            results.append(result)
            print(f"\r{i_progress}/{len(layer_sizes_list) * len(learning_rates_list)}", end = "")
            i_progress += 1
        
    print() 
    return results

In [38]:
epochs_num = 10
metrics = [("Accuracy", accuracy_score, {}), ("F1-score", f1_score, {"average" : "macro"})]
layer_sizes_list = [[300, 64, 32], [300, 64, 64], [300, 128, 64, 32], [300, 64, 64, 64]]
learning_rates_list = [0.001, 0.0001]

In [153]:
word2vec_df = word2vec_df.dropna()
word2vec_train_dl, word2vec_val_dl, word2vec_test_dl = get_data_loaders(SimpleDataset, word2vec_df["ttext_vectors"], word2vec_df["ttype"])

fasttext_df = fasttext_df.dropna()
fasttext_train_dl, fasttext_val_dl, fasttext_test_dl = get_data_loaders(SimpleDataset, fasttext_df["ttext_vectors"], fasttext_df["ttype"])

embeddings_df = embeddings_df.dropna()
embeddings_train_dl, embeddings_val_dl, embeddings_test_dl = get_data_loaders(EmbeddingsDataset, embeddings_df["ttext_vectors"], embeddings_df["ttype"])

In [154]:
word2vec_results = get_results("word2vec", SimpleNetwork, word2vec_train_dl, word2vec_val_dl, word2vec_test_dl, layer_sizes_list, learning_rates_list, epochs_num, metrics)
fasttext_results = get_results("fasttext", SimpleNetwork, fasttext_train_dl, fasttext_val_dl, fasttext_test_dl, layer_sizes_list, learning_rates_list, epochs_num, metrics)
embeddings_results = get_results("nn.Embedding", EmbeddingsNetwork, embeddings_train_dl, embeddings_val_dl, embeddings_test_dl, layer_sizes_list, learning_rates_list, epochs_num, metrics)

8/8
8/8
8/8


In [169]:
results = pd.concat([pd.DataFrame(word2vec_results), pd.DataFrame(fasttext_results), pd.DataFrame(embeddings_results)])[["method", "layer_sizes", "learning_rate", "Accuracy", "F1-score"]].reset_index(drop = True).round(4)

In [170]:
results.sort_values(by = "Accuracy", ascending = False)

,method,layer_sizes,learning_rate,Accuracy,F1-score
22,nn.Embedding,"[64, 64, 64]",0.0010,0.6615,0.6609
20,nn.Embedding,"[128, 64, 32]",0.0010,0.6525,0.6495
16,nn.Embedding,"[64, 32]",0.0010,0.6490,0.6476
18,nn.Embedding,"[64, 64]",0.0010,0.6435,0.6431
19,nn.Embedding,"[64, 64]",0.0001,0.5555,0.5497
2,word2vec,"[64, 64]",0.0010,0.5223,0.4312
0,word2vec,"[64, 32]",0.0010,0.5008,0.3398
1,word2vec,"[64, 32]",0.0001,0.5003,0.3334
15,fasttext,"[64, 64, 64]",0.0001,0.5003,0.3334
14,fasttext,"[64, 64, 64]",0.0010,0.5003,0.3334


In [171]:
results.sort_values(by = "F1-score", ascending = False)

,method,layer_sizes,learning_rate,Accuracy,F1-score
22,nn.Embedding,"[64, 64, 64]",0.0010,0.6615,0.6609
20,nn.Embedding,"[128, 64, 32]",0.0010,0.6525,0.6495
16,nn.Embedding,"[64, 32]",0.0010,0.6490,0.6476
18,nn.Embedding,"[64, 64]",0.0010,0.6435,0.6431
19,nn.Embedding,"[64, 64]",0.0001,0.5555,0.5497
2,word2vec,"[64, 64]",0.0010,0.5223,0.4312
21,nn.Embedding,"[128, 64, 32]",0.0001,0.4920,0.4078
0,word2vec,"[64, 32]",0.0010,0.5008,0.3398
1,word2vec,"[64, 32]",0.0001,0.5003,0.3334
15,fasttext,"[64, 64, 64]",0.0001,0.5003,0.3334


## Выводы

Лучший результат и по accuracy, и по F1-мере показывает нейросеть из 3 слоёв по 64 нейрона с learning rate = 0.001 и nn.Embedding в качестве признаков. Другие модели, показывающие лучшие результаты, также используют nn.Embedding в качестве признаков и learning rate = 0.001. Конфигурация слоёв при этом имеет наименьшее влияние, что можно объяснить тем, что обучение производилось всегда на 10 эпохах, вне зависимости от выбранных гиперпараметров.